# 07 — EDA: pLLM embeddings (TEM-1 / Firnberg)

This EDA follows the same flow as the project's root `ML EDA Notebook` — understand the data,
check quality, look at distributions, hunt outliers, then relationships — but the inputs are ESM
**embedding-delta** features (the arm-3 supervised model trains on these). A raw embedding matrix is
~27k columns, so we can't histogram every dimension or draw a 27k-wide correlation heatmap. Instead,
following the same logic the root notebook applies to `surprisal`/`DMS_score`, we reduce each model's
**`delta_site`** block (the primary variant-specific feature, D035) to its top few **principal
components** and treat those PCs as the numeric columns. PCA here is unsupervised and fit on the whole
dataset — this is EDA, not modeling, so there is no train/test fold (the in-fold discipline lives in
`08`, D037). Each PC is labeled by its model and variance explained.

**Inputs read from disk** (never in-memory state, per CLAUDE.md):
- the processed modeling dataset `data/processed/pllm_embeddings/modeling_dataset.parquet` (labels +
  embedding features in one table, assembled like the `02`/`04` datasets). Absent → PREVIEW banner,
  clean stop.


In [ ]:
# self-contained: resolve project root via .projectroot, read from disk
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'.projectroot').exists())
sys.path.insert(0, str(root)); from paths import *

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
try:
    from IPython.display import display
except Exception:
    def display(x): print(x)

SEED = 42
N_PC = 4          # top principal components per model's delta_site block -> the "numeric columns"
PRIMARY_BLOCK = "delta_site"   # D035 primary feature; the block we build PCs from
MODEL_ORDER = ["esm1b","esm1v","esm2_150m","esm2_650m","esm2_3b","esmc_300m","esmc_600m"]
MODEL_LABEL = {"esm1b":"ESM-1b 650M","esm1v":"ESM-1v 650M","esm2_150m":"ESM-2 150M",
               "esm2_650m":"ESM-2 650M","esm2_3b":"ESM-2 3B","esmc_300m":"ESM-C 300M","esmc_600m":"ESM-C 600M"}
BLOCKS = ["delta_site","delta_pooled","delta_local"]
# project palette: blues, greens, purples, dark pinks only
MODEL_COLORS = {"esm1b":"#2c6fbb","esm1v":"#1f4e8c","esm2_150m":"#57a773","esm2_650m":"#2e8b57",
                "esm2_3b":"#1d6b3f","esmc_300m":"#7a4fa3","esmc_600m":"#b03060"}
PAL = {"blue":"#2c6fbb","green":"#2e8b57","purple":"#7a4fa3","pink":"#b03060","grey":"#9aa0a6"}
sns.set_theme(style="whitegrid")

NBNAME = "07_EDA_pllm_embeddings"
FIGDIR = FIGURES/NBNAME; FIGDIR.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)
print("root:", root, "| seed:", SEED, "| PCs per model:", N_PC)


## Load and build the PC feature table

Read the processed modeling dataset, then reduce each model's `delta_site` block to its top few PCs.
These PCs are the "numeric columns" for the rest of the EDA — the stand-in for `surprisal` / `DMS_score`
in the root notebook.

In [ ]:
DS_PATH = PROCESSED/"pllm_embeddings"/"modeling_dataset.parquet"
PREVIEW = not DS_PATH.exists()
if PREVIEW:
    print("="*78)
    print("PREVIEW MODE — processed embedding dataset not found at:", DS_PATH)
    print("Run 07a_pllm_embedding_extraction (or the Colab twin) + the dataset-assembly step, then re-run.")
    print("="*78)
else:
    raw = pd.read_parquet(DS_PATH).sort_values(["position","mut_aa"]).reset_index(drop=True)
    assert raw.seq_id.is_unique, "duplicate seq_id"
    ID_LAB = ["seq_id","wt_aa","mut_aa","position","DMS_score","DMS_score_bin"]
    feat_cols = [c for c in raw.columns if c not in ID_LAB]
    present_models = [m for m in MODEL_ORDER
                      if any(c.startswith(f"{m}__{PRIMARY_BLOCK}__") for c in feat_cols)]
    # build top-N_PC PCs per model's delta_site block (unsupervised, whole dataset -- EDA, not modeling)
    pc_cols=[]; var_expl={}
    df = raw[ID_LAB].copy()
    for m in present_models:
        cols=[c for c in feat_cols if c.startswith(f"{m}__{PRIMARY_BLOCK}__")]
        Z = StandardScaler().fit_transform(raw[cols].values.astype(np.float32))
        p = PCA(n_components=N_PC, random_state=SEED).fit(Z)
        T = p.transform(Z)
        for k in range(N_PC):
            name=f"{m}_PC{k+1}"; df[name]=T[:,k]; pc_cols.append(name)
            var_expl[name]=float(p.explained_variance_ratio_[k])
    print(f"variants={len(df)} | models={present_models} | PC columns={len(pc_cols)} ({N_PC} per model)")
    print("wt_seq len 286; positions", df.position.min(), "-", df.position.max())


## Understanding our data

Shape first, then a peek at the rows. Same discipline as the root notebook — `.head()` can lie (rows
are sorted by position/mutation here), so we treat it as a sanity glance, not a summary.

In [ ]:
if not PREVIEW:
    print("rows, columns:", df.shape)
    print("PC columns:", len(pc_cols), "| id/label columns:", len(ID_LAB))


In [ ]:
if not PREVIEW:
    display(df[ID_LAB + pc_cols[:4]].head(3))


**Split the columns by type up front.** The PCs and `DMS_score`/`position` are numeric; `wt_aa`/`mut_aa`
are categorical; `DMS_score_bin` is the boolean-like label. Numeric and categorical get different
treatment downstream.

In [ ]:
if not PREVIEW:
    num_cols = pc_cols + ["DMS_score","position"]
    cat_cols = ["wt_aa","mut_aa"]
    print("numeric:", len(num_cols), "cols (", N_PC, "PCs x", len(present_models), "models + DMS_score + position )")
    print("categorical:", cat_cols)
    print("label:", "DMS_score_bin", "| balance:", raw.DMS_score_bin.value_counts().to_dict())


### Data Quality — Trust, But Verify (maybe don't even trust, just verify)

The embeddings crossed from an external GPU run (Colab), so nothing is trusted until checked:
missingness, duplicates, cardinality, and whether the numbers make physical sense. A NaN here would
mean a model or position was silently skipped during extraction.

In [ ]:
if not PREVIEW:
    missing = df[num_cols].isna().sum()
    print("total missing across numeric columns:", int(missing.sum()), " (must be 0)")
    assert missing.sum()==0, "non-finite PC/label values"
    # leakage guard (CONVENTIONS.md): no id/label column leaked into the feature view, and no single
    # raw embedding dim tracks the label almost perfectly (chunked, so we never build a DxD matrix)
    y = raw.DMS_score_bin.values.astype(int); yz=(y-y.mean()); yz/=np.linalg.norm(yz)+1e-12
    X = raw[feat_cols].values.astype(np.float32); mx=0.0
    for s in range(0,X.shape[1],4096):
        b=X[:,s:s+4096].copy(); b-=b.mean(0); n=np.linalg.norm(b,axis=0)+1e-12
        mx=max(mx, float(np.nanmax(np.abs((b*yz[:,None]).sum(0)/n))))
    print(f"max |corr(raw embedding dim, label)| = {mx:.3f}  -> {'OK' if mx<0.99 else 'LEAKAGE'}")
    assert mx<0.99, "an embedding dim tracks the label almost perfectly — leakage?"


In [ ]:
if not PREVIEW:
    print("exact duplicate rows (should be 0 — every variant is unique):", int(df.duplicated(subset=pc_cols).sum()))
    print("duplicate seq_ids:", int(df.seq_id.duplicated().sum()))


**Does the data make physical sense?** PC scores are standardized-space projections, so each should be
roughly mean-zero with no absurd values; `DMS_score` inside its known range; `position` within the
protein length (24–286). `describe()` is the quick check.

In [ ]:
if not PREVIEW:
    display(df[pc_cols[:8]+["DMS_score","position"]].describe().round(2))


## Always Check Distributions

Histogram every PC. Unlike raw `surprisal` (a long right tail), PCA scores are usually centered and
roughly symmetric by construction — but a heavy tail or a bimodal PC is worth seeing, since it hints at
structure the classifier can use.

In [ ]:
if not PREVIEW:
    show = pc_cols[:min(len(pc_cols), 16)]
    axes = df[show].hist(bins=30, figsize=(13,8), color="#bcd4ec", edgecolor="#7a4fa3")
    fig = axes[0,0].get_figure(); fig.suptitle("Distributions of top PCs (delta_site)", y=1.0)
    fig.tight_layout(); fig.savefig(FIGDIR/"pc_histograms.pdf",bbox_inches="tight")
    fig.savefig(FIGDIR/"pc_histograms.png",bbox_inches="tight",dpi=150); plt.show()


**Skew** measures asymmetry. ≈0 is symmetric; large magnitude means a long tail. Many methods assume
roughly symmetric inputs; a strongly skewed PC flags a direction dominated by a few extreme variants.

In [ ]:
if not PREVIEW:
    display(df[pc_cols].skew().round(2).sort_values(key=np.abs, ascending=False).to_frame("skew").head(10))


Since we'll be predicting `DMS_score_bin`, pay special attention to the continuous `DMS_score` it comes
from — its distribution and the binarization threshold.

In [ ]:
if not PREVIEW:
    fig, ax = plt.subplots(1,2, figsize=(11,4))
    ax[0].hist(raw.DMS_score, bins=40, color="#bcd4ec", edgecolor="#7a4fa3")
    ax[0].set_title("DMS_score"); ax[0].set_xlabel("DMS_score")
    raw.DMS_score_bin.value_counts().sort_index().plot(kind="bar", ax=ax[1],
        color=[PAL["pink"],PAL["green"]]); ax[1].set_title("DMS_score_bin (0=non-func, 1=func)")
    ax[1].set_xticklabels(["non-func","func"], rotation=0)
    fig.tight_layout(); fig.savefig(FIGDIR/"target_distribution.pdf",bbox_inches="tight")
    fig.savefig(FIGDIR/"target_distribution.png",bbox_inches="tight",dpi=150); plt.show()


Boxplots summarize spread and flag outliers at a glance — one box per model's PC1, so we can see which
models produce wider / more outlier-heavy leading directions.

In [ ]:
if not PREVIEW:
    pc1 = [f"{m}_PC1" for m in present_models if f"{m}_PC1" in df.columns]
    fig, ax = plt.subplots(figsize=(11,4.6))
    bp = ax.boxplot([df[c].values for c in pc1], tick_labels=[MODEL_LABEL[m] for m in present_models],
                    patch_artist=True, showfliers=True, flierprops=dict(marker=".",markersize=3,alpha=.3))
    for patch,m in zip(bp["boxes"],present_models): patch.set_facecolor(MODEL_COLORS[m]); patch.set_alpha(.7)
    ax.set_ylabel("PC1 score"); ax.set_title("PC1 spread and outliers by model (delta_site)")
    plt.xticks(rotation=25, ha="right")
    fig.tight_layout(); fig.savefig(FIGDIR/"pc1_boxplots.pdf",bbox_inches="tight")
    fig.savefig(FIGDIR/"pc1_boxplots.png",bbox_inches="tight",dpi=150); plt.show()


## Looking at categorical variables

The mutation identity columns are categorical. `wt_aa` / `mut_aa` value counts show which residues the
DMS panel sampled — the same categorical signal the AA-identity baseline (02) leaned on.

In [ ]:
if not PREVIEW:
    fig, ax = plt.subplots(1,2, figsize=(13,4))
    raw.wt_aa.value_counts().reindex(list("ACDEFGHIKLMNPQRSTVWY")).plot(kind="bar", ax=ax[0],
        color=PAL["blue"]); ax[0].set_title("wild-type residue (wt_aa)"); ax[0].set_ylabel("count")
    raw.mut_aa.value_counts().reindex(list("ACDEFGHIKLMNPQRSTVWY")).plot(kind="bar", ax=ax[1],
        color=PAL["green"]); ax[1].set_title("substituted residue (mut_aa)")
    fig.tight_layout(); fig.savefig(FIGDIR/"residue_counts.pdf",bbox_inches="tight")
    fig.savefig(FIGDIR/"residue_counts.png",bbox_inches="tight",dpi=150); plt.show()


## Relationships between variables

The point of the arm: do these embedding directions track function? With 4,783 points a raw scatter
piles up, so we use a **hexbin** of each model's PC1 against `DMS_score`, colored by density, to see
whether the leading embedding direction separates functional from dead variants.

In [ ]:
if not PREVIEW:
    best_pc1 = max((f"{m}_PC1" for m in present_models if f"{m}_PC1" in df.columns),
                   key=lambda c: max(roc_auc_score(raw.DMS_score_bin, df[c]),
                                     roc_auc_score(raw.DMS_score_bin, -df[c])))
    fig, ax = plt.subplots(figsize=(7.5,6))
    hb = ax.hexbin(df[best_pc1], raw.DMS_score, gridsize=40, cmap="BuPu", mincnt=1)
    fig.colorbar(hb, ax=ax, label="variant count")
    ax.set_xlabel(best_pc1); ax.set_ylabel("DMS_score")
    ax.set_title(f"Density: {best_pc1} vs DMS_score (strongest single PC)")
    fig.tight_layout(); fig.savefig(FIGDIR/"hexbin_pc_vs_dms.pdf",bbox_inches="tight")
    fig.savefig(FIGDIR/"hexbin_pc_vs_dms.png",bbox_inches="tight",dpi=150); plt.show()
    print("strongest PC1 by |AUC|:", best_pc1)


**Correlation** quantifies how two numbers move together. **Pearson** catches linear association;
**Spearman** catches monotonic (rank) association even when curved. We correlate every PC against
`DMS_score`; where the two disagree, the PC relates to function nonlinearly.

In [ ]:
if not PREVIEW:
    pear = df[pc_cols].corrwith(raw.DMS_score, method="pearson")
    spear = df[pc_cols].corrwith(raw.DMS_score, method="spearman")
    corr_tbl = pd.DataFrame({"pearson_vs_DMS":pear, "spearman_vs_DMS":spear})
    corr_tbl["abs_gap"]=(corr_tbl.pearson_vs_DMS-corr_tbl.spearman_vs_DMS).abs()
    display(corr_tbl.reindex(corr_tbl.spearman_vs_DMS.abs().sort_values(ascending=False).index).round(3).head(12))


A heatmap of the PC correlation matrix shows every pairwise relationship at once — including whether
different models' PCs are redundant (dark off-diagonal blocks) or capture distinct directions.

In [ ]:
if not PREVIEW:
    C = df[pc_cols].corr()
    fig, ax = plt.subplots(figsize=(9,7.5))
    sns.heatmap(C, cmap="PuOr", center=0, vmin=-1, vmax=1, square=True,
                cbar_kws={"label":"Pearson r"}, ax=ax, xticklabels=True, yticklabels=True)
    ax.set_title("PC-PC correlation across models"); ax.tick_params(labelsize=6)
    fig.tight_layout(); fig.savefig(FIGDIR/"pc_correlation_heatmap.pdf",bbox_inches="tight")
    fig.savefig(FIGDIR/"pc_correlation_heatmap.png",bbox_inches="tight",dpi=150); plt.show()


Numeric-vs-categorical relationships need grouped distributions. Split the strongest PC by the
**functional class** (dead vs functional) — if the two distributions separate, that PC carries usable
functional signal before any training. This is the embedding analog of the root notebook's active-site
/ secondary-structure split.

In [ ]:
if not PREVIEW:
    df["functional_class"] = np.where(raw.DMS_score_bin.values==1, "functional", "dead")
    fig, ax = plt.subplots(1,2, figsize=(13,5))
    for cls,col in [("functional",PAL["green"]),("dead",PAL["pink"])]:
        sub=df[df.functional_class==cls][best_pc1]
        ax[0].hist(sub, bins=40, alpha=.55, label=cls, color=col)
    ax[0].set_title(f"{best_pc1} by functional class"); ax[0].set_xlabel(best_pc1); ax[0].legend(frameon=False)
    sns.boxplot(data=df, x="functional_class", y=best_pc1, ax=ax[1],
                palette={"functional":PAL["green"],"dead":PAL["pink"]}, order=["dead","functional"])
    ax[1].set_title(f"{best_pc1} split by class")
    fig.tight_layout(); fig.savefig(FIGDIR/"pc_by_class.pdf",bbox_inches="tight")
    fig.savefig(FIGDIR/"pc_by_class.png",bbox_inches="tight",dpi=150); plt.show()
    # single-feature AUC of that PC (the pre-training separation floor)
    a=max(roc_auc_score(raw.DMS_score_bin,df[best_pc1]), roc_auc_score(raw.DMS_score_bin,-df[best_pc1]))
    print(f"{best_pc1} single-feature ROC-AUC vs functional label: {a:.3f}")


Finally, a **pairplot** of the top PCs of the single best-separating model, colored by functional class
— every pair as a scatter, each PC's distribution on the diagonal. It shows which embedding directions
actually pull the two classes apart, the overview the whole arm rests on.

In [ ]:
if not PREVIEW:
    best_model = best_pc1.rsplit("_PC",1)[0]
    cols=[f"{best_model}_PC{k+1}" for k in range(min(N_PC,4)) if f"{best_model}_PC{k+1}" in df.columns]
    pp = sns.pairplot(df[cols+["functional_class"]], hue="functional_class",
                      palette={"functional":PAL["green"],"dead":PAL["pink"]},
                      plot_kws=dict(s=8, alpha=.35, edgecolor="none"), diag_kind="hist", corner=True)
    pp.figure.suptitle(f"Top PCs of {MODEL_LABEL.get(best_model,best_model)} by functional class", y=1.02)
    pp.figure.savefig(FIGDIR/"pc_pairplot.pdf",bbox_inches="tight")
    pp.figure.savefig(FIGDIR/"pc_pairplot.png",bbox_inches="tight",dpi=150); plt.show()
    print("pairplot model:", best_model)


## Save tables

In [ ]:
if not PREVIEW:
    # variance explained per PC + its correlation/AUC with the label
    rows=[]
    for c in pc_cols:
        m=c.rsplit("_PC",1)[0]
        rows.append(dict(pc=c, model=m, var_explained=round(var_expl[c],4),
            pearson_vs_DMS=round(float(df[c].corr(raw.DMS_score)),4),
            spearman_vs_DMS=round(float(df[c].corr(raw.DMS_score,method="spearman")),4),
            single_pc_auc=round(float(max(roc_auc_score(raw.DMS_score_bin,df[c]),
                                          roc_auc_score(raw.DMS_score_bin,-df[c]))),4)))
    pc_summary=pd.DataFrame(rows).sort_values("single_pc_auc",ascending=False)
    pc_summary.to_csv(TABLES/f"{NBNAME}_pc_summary.csv", index=False)
    corr_tbl.round(4).to_csv(TABLES/f"{NBNAME}_pc_dms_correlation.csv")
    print("saved tables:")
    for f in sorted(TABLES.glob(f"{NBNAME}_*.csv")): print("  ", f.name)
    print("saved figures:")
    for f in sorted(FIGDIR.glob("*.pdf")): print("  ", f.name)
    print("\ntop 5 PCs by single-feature AUC:")
    print(pc_summary.head(5).to_string(index=False))
else:
    print("PREVIEW mode — nothing written.")


## Summary Goals of EDA

**Understand the data** — the embedding-delta features reduce to a handful of PCs per model; the
`pc_summary` table ranks them by how much variance they carry and how well each separates functional
from dead.

**Detect outliers** — the PC boxplots flag models with heavy-tailed leading directions; no missing or
non-finite values survived the quality check.

**Formulate hypotheses** — the best single PC's class separation (its ROC-AUC) is the pre-training
signal floor. If a single embedding direction already separates the classes, the arm-3 supervised
model should comfortably beat it; if not, the signal is distributed across many dimensions and the
classifier has to combine them.

**Check assumptions** — the PC-PC correlation heatmap shows how redundant the model ladder is: models
whose PCs correlate strongly carry overlapping signal, so a subset may suffice in `08`.

**Limitation (D039).** These embeddings are sequence-only and shaped by foldability/stability, so — like
the zero-shot scores — they can under-represent a catalytic-but-stable knockout. This EDA measures how
separable the *training* signal is; whether that signal is the clinically right one is what the ESMFold
structural features (D038) and the wet-lab panel test.